# 05 — Test-Year Target Grid Alignment (FIXED)

This replaces the old workflow that aligned **all years** to the first NDVI raster.

**Corrected logic**
- Training/validation rasters remain native and are NOT processed here.
- Only the independent spatial prediction year (2022) is resampled to one target grid.
- Raw source rasters are used directly; no "aligned → aligned again" chain.
- No nearest-value filling of large NoData regions.
- No edge clamping.
- Bilinear interpolation is used for continuous variables.
- Output NoData remains NoData.

In [1]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [2]:
import re, math
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin
from rasterio.features import geometry_mask
import geopandas as gpd
from pyproj import CRS

TEST_YEAR = 2022
TARGET_RES_DEG = 0.005
DST_CRS = "EPSG:4326"
DST_NODATA = -9999.0

PRECIP_PRODUCTS = ["CCS","PDIR","GSMaP_MVK","CDR","CHIRPS","IMERG","GSMaP_Gauge_v7","ERA5"]
LAND_DYNAMIC = ["NDVI","LST_Day"]

def parse_ym(name):
    stem = Path(name).stem
    for pat in [r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
                r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"]:
        m = re.search(pat, stem)
        if m:
            return int(m.group(1)), int(m.group(2))
    return None

def list_rasters(folder):
    return sorted([*folder.rglob("*.tif"), *folder.rglob("*.tiff")])

def monthly_map(folder):
    out = {}
    for p in list_rasters(folder):
        ym = parse_ym(p.name)
        if ym:
            if ym in out:
                raise ValueError(f"Duplicate month {ym} in {folder}")
            out[ym] = p
    return out

def source_crs_or_verified_wgs84(src, path):
    if src.crs is not None:
        return src.crs
    b = src.bounds
    geographic_bounds = (-180 <= b.left <= 180 and -180 <= b.right <= 180 and
                         -90 <= b.bottom <= 90 and -90 <= b.top <= 90)
    if geographic_bounds:
        warnings.warn(f"{path.name}: missing CRS but geographic-looking bounds; assuming EPSG:4326.")
        return CRS.from_epsg(4326)
    raise ValueError(f"{path}: CRS missing and bounds are not safely interpretable as lon/lat.")

In [3]:
# Find study boundary and convert it to WGS84.
boundary_dir = RAW_DIR / "boundary"
boundary_candidates = [
    *boundary_dir.rglob("*.gpkg"),
    *boundary_dir.rglob("*.shp"),
    *boundary_dir.rglob("*.geojson"),
]
if not boundary_candidates:
    raise FileNotFoundError("No boundary vector found under data/raw/boundary")

boundary_path = boundary_candidates[0]
gdf = gpd.read_file(boundary_path)
if gdf.crs is None:
    raise ValueError("Study boundary has no CRS.")
gdf = gdf.to_crs(DST_CRS)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
if gdf.empty:
    raise ValueError("Boundary contains no valid geometry.")

minx, miny, maxx, maxy = gdf.total_bounds

# Snap outward to target grid.
left   = math.floor(minx / TARGET_RES_DEG) * TARGET_RES_DEG
right  = math.ceil (maxx / TARGET_RES_DEG) * TARGET_RES_DEG
bottom = math.floor(miny / TARGET_RES_DEG) * TARGET_RES_DEG
top    = math.ceil (maxy / TARGET_RES_DEG) * TARGET_RES_DEG

width = int(round((right-left)/TARGET_RES_DEG))
height = int(round((top-bottom)/TARGET_RES_DEG))
transform = from_origin(left, top, TARGET_RES_DEG, TARGET_RES_DEG)

inside_mask = geometry_mask(
    gdf.geometry,
    transform=transform,
    invert=True,
    out_shape=(height,width),
    all_touched=False,
)

print("Boundary:", boundary_path)
print("Target grid:", width, "x", height)
print("Bounds:", (left,bottom,right,top))
print("Grid spacing:", TARGET_RES_DEG, "degree")

Boundary: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\boundary\Khulna.shp
Target grid: 105 x 271
Bounds: (89.235, 21.66, 89.76, 23.015)
Grid spacing: 0.005 degree


In [5]:
# ============================================================
# CANONICAL STATIC PREDICTORS
# ============================================================

def choose_static(folder_name, preferred_names):
    folder = RAW_DIR / "predictors" / folder_name
    files = list_rasters(folder)

    if not files:
        raise FileNotFoundError(f"No raster found for {folder_name}")

    # Prefer exact known filenames
    d = {p.name.lower(): p for p in files}

    for n in preferred_names:
        if n.lower() in d:
            return d[n.lower()]

    # Avoid obvious intermediate files
    clean = [
        p for p in files
        if not any(
            x in p.stem.lower()
            for x in ["clip", "tmp", "temp", "aligned", "resampl"]
        )
    ]

    if len(clean) == 1:
        return clean[0]

    if len(files) == 1:
        return files[0]

    raise ValueError(
        f"Ambiguous static predictor {folder_name}:\n" +
        "\n".join(str(p) for p in files)
    )


DEM_PATH = choose_static(
    "DEM",
    ["Khulna_SRTM_DEM.tif", "DEM.tif"]
)

DFS_PATH = choose_static(
    "Distance_Sea",
    ["Distance_Sea.tif", "distance_to_sea.tif"]
)

print("DEM:", DEM_PATH)
print("Distance to Sea:", DFS_PATH)


# ============================================================
# ACTUAL PRECIPITATION FOLDER MAPPING
# ============================================================

PRECIP_FOLDERS = {
    "CCS": "CCS",
    "PDIR": "PDIR",
    "GSMaP_MVK": "GSMaP_MVK",
    "CDR": "CDR",
    "CHIRPS": "CHIRPS_TIFF_2017_2022",
    "IMERG": "IMERG_Monthly",
    "GSMaP_Gauge": "GSMaP_Gauge_v7",
    "ERA5": "ERA5_TIFF",
}


# ============================================================
# PREPARE 2022 PRECIPITATION SOURCES
# ============================================================

sources = {}

for product, folder_name in PRECIP_FOLDERS.items():

    folder = RAW_DIR / "precipitation" / folder_name

    if not folder.exists():
        raise FileNotFoundError(
            f"Missing precipitation folder:\n{folder}"
        )

    mm = monthly_map(folder)

    for month in range(1, 13):

        if (TEST_YEAR, month) not in mm:
            raise FileNotFoundError(
                f"Missing {product} raster for "
                f"{TEST_YEAR}-{month:02d}"
            )

        sources[(product, month)] = mm[(TEST_YEAR, month)]


# ============================================================
# PREPARE 2022 DYNAMIC LAND SOURCES
# ============================================================

for pred in LAND_DYNAMIC:

    folder = RAW_DIR / "predictors" / pred

    if not folder.exists():
        raise FileNotFoundError(
            f"Missing predictor folder:\n{folder}"
        )

    mm = monthly_map(folder)

    for month in range(1, 13):

        if (TEST_YEAR, month) not in mm:
            raise FileNotFoundError(
                f"Missing {pred} raster for "
                f"{TEST_YEAR}-{month:02d}"
            )

        sources[(pred, month)] = mm[(TEST_YEAR, month)]


# ============================================================
# ADD STATIC LAND PREDICTORS
# ============================================================

sources[("DEM", 0)] = DEM_PATH
sources[("Distance_Sea", 0)] = DFS_PATH


# ============================================================
# FINAL SOURCE CHECK
# ============================================================

print("\n========================================")
print("SOURCE PREPARATION COMPLETE")
print("========================================")

print("Total sources prepared:", len(sources))

print("\nExpected:")
print("8 precipitation products × 12 months = 96")
print("2 dynamic land predictors × 12 months = 24")
print("2 static predictors = 2")
print("Total expected = 122")

if len(sources) != 122:
    print("\nWARNING: Expected 122 sources, found:", len(sources))
else:
    print("\n✅ All 122 required sources found successfully.")

DEM: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\DEM\Khulna_SRTM_DEM.tif
Distance to Sea: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\Distance_Sea\Distance_Sea.tif

SOURCE PREPARATION COMPLETE
Total sources prepared: 122

Expected:
8 precipitation products × 12 months = 96
2 dynamic land predictors × 12 months = 24
2 static predictors = 2
Total expected = 122

✅ All 122 required sources found successfully.


In [11]:
# ============================================================
# 05 — TEST YEAR TARGET GRID ALIGNMENT (FULL FIXED)
# Khulna Precipitation Downscaling
# ============================================================

from pathlib import Path
import re
import math
import warnings

import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd

from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from rasterio.features import geometry_mask


# ============================================================
# 1. FIND PROJECT ROOT
# ============================================================

def find_project_root(start=None):

    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:

        if (candidate / "data").exists():

            return candidate

    raise FileNotFoundError(
        "Project root not found. "
        "Run this notebook from inside the repository."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_ROOT =", PROJECT_ROOT)


# ============================================================
# 2. SETTINGS
# ============================================================

TEST_YEAR = 2022

TARGET_RES_DEG = 0.005

DST_NODATA = -9999.0

# Avoid EPSG database lookup
WGS84_PROJ = "+proj=longlat +datum=WGS84 +no_defs"


# ============================================================
# 3. PRECIPITATION FOLDER MAPPING
# ============================================================

PRECIP_FOLDERS = {

    "CCS":
        "CCS",

    "PDIR":
        "PDIR",

    "GSMaP_MVK":
        "GSMaP_MVK",

    "CDR":
        "CDR",

    "CHIRPS":
        "CHIRPS_TIFF_2017_2022",

    "IMERG":
        "IMERG_Monthly",

    "GSMaP_Gauge":
        "GSMaP_Gauge_v7",

    "ERA5":
        "ERA5_TIFF"
}


# ============================================================
# 4. DYNAMIC LAND VARIABLES
# ============================================================

LAND_DYNAMIC = [

    "NDVI",
    "LST_Day"

]


# ============================================================
# 5. HELPER — PARSE YEAR/MONTH
# ============================================================

def parse_ym(name):

    stem = Path(name).stem

    patterns = [

        r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",

        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"
    ]

    for pat in patterns:

        m = re.search(
            pat,
            stem
        )

        if m:

            return (
                int(m.group(1)),
                int(m.group(2))
            )

    return None


# ============================================================
# 6. HELPER — LIST RASTERS
# ============================================================

def list_rasters(folder):

    if not folder.exists():

        return []

    return sorted([

        *folder.rglob("*.tif"),

        *folder.rglob("*.tiff")

    ])


# ============================================================
# 7. HELPER — MONTHLY MAP
# ============================================================

def monthly_map(folder):

    out = {}

    files = list_rasters(
        folder
    )

    for p in files:

        ym = parse_ym(
            p.name
        )

        if ym:

            if ym in out:

                raise ValueError(

                    f"Duplicate raster for {folder.name} {ym}:\n"
                    f"{out[ym]}\n"
                    f"{p}"
                )

            out[ym] = p

    return out


# ============================================================
# 8. HELPER — CHOOSE STATIC RASTER
# ============================================================

def choose_static(
    folder_name,
    preferred_names
):

    folder = (
        RAW_DIR
        / "predictors"
        / folder_name
    )

    files = list_rasters(
        folder
    )

    if not files:

        raise FileNotFoundError(
            f"No raster found for {folder_name}"
        )

    name_lookup = {

        p.name.lower(): p

        for p in files
    }


    # Prefer exact filename
    for name in preferred_names:

        if name.lower() in name_lookup:

            return name_lookup[
                name.lower()
            ]


    # Avoid intermediate rasters
    clean = [

        p for p in files

        if not any(

            keyword in p.stem.lower()

            for keyword in [

                "clip",
                "tmp",
                "temp",
                "aligned",
                "resampl"

            ]
        )

    ]


    if len(clean) == 1:

        return clean[0]


    if len(files) == 1:

        return files[0]


    raise ValueError(

        f"Ambiguous static predictor "
        f"{folder_name}:\n"

        + "\n".join(
            str(p)
            for p in files
        )
    )


# ============================================================
# 9. FIX / RESOLVE SOURCE CRS
# ============================================================

def source_crs_or_verified_wgs84(
    src,
    path
):

    b = src.bounds


    # --------------------------------------------------------
    # Do coordinates look like longitude/latitude?
    # --------------------------------------------------------

    geographic_bounds = (

        -180 <= b.left <= 180

        and

        -180 <= b.right <= 180

        and

        -90 <= b.bottom <= 90

        and

        -90 <= b.top <= 90
    )


    # --------------------------------------------------------
    # CASE 1: No CRS at all
    # --------------------------------------------------------

    if src.crs is None:

        if geographic_bounds:

            print(

                f"WARNING: {path.name} has no CRS -> "
                "using WGS84 because bounds are lon/lat."
            )

            return WGS84_PROJ


        raise ValueError(

            f"{path.name}: CRS missing and "
            "bounds are not lon/lat."
        )


    # --------------------------------------------------------
    # CASE 2: CRS exists — check whether usable
    # --------------------------------------------------------

    try:

        is_geographic = bool(
            src.crs.is_geographic
        )

        is_projected = bool(
            src.crs.is_projected
        )

    except Exception:

        is_geographic = False

        is_projected = False


    # --------------------------------------------------------
    # Normal geographic/projected CRS
    # --------------------------------------------------------

    if is_geographic or is_projected:

        return src.crs


    # --------------------------------------------------------
    # CASE 3:
    # EngineeringCRS / LOCAL_CS / unusable CRS
    # but bounds are clearly lon/lat
    # --------------------------------------------------------

    if geographic_bounds:

        print(

            f"WARNING: {path.name} has invalid/Engineering CRS -> "
            "forcing WGS84 because coordinates are geographic."
        )

        return WGS84_PROJ


    # --------------------------------------------------------
    # Unsafe case
    # --------------------------------------------------------

    raise ValueError(

        f"{path.name}: CRS is unusable and "
        "bounds are not safely identifiable as lon/lat."
    )


# ============================================================
# 10. CANONICAL STATIC PREDICTORS
# ============================================================

DEM_PATH = choose_static(

    "DEM",

    [
        "Khulna_SRTM_DEM.tif",
        "DEM.tif"
    ]
)


DFS_PATH = choose_static(

    "Distance_Sea",

    [
        "Distance_Sea.tif",
        "distance_to_sea.tif"
    ]
)


print(
    "\nCanonical DEM:"
)

print(
    DEM_PATH
)


print(
    "\nCanonical Distance to Sea:"
)

print(
    DFS_PATH
)


# ============================================================
# 11. FIND KHULNA BOUNDARY
# ============================================================

boundary_dir = (
    RAW_DIR
    / "boundary"
)


boundary_candidates = [

    *boundary_dir.rglob("*.shp"),

    *boundary_dir.rglob("*.gpkg"),

    *boundary_dir.rglob("*.geojson")

]


if not boundary_candidates:

    raise FileNotFoundError(

        "No boundary vector found under "
        "data/raw/boundary"
    )


boundary_path = (
    boundary_candidates[0]
)


print(
    "\nBoundary file:"
)

print(
    boundary_path
)


# ============================================================
# 12. READ BOUNDARY
# ============================================================

gdf = gpd.read_file(
    boundary_path
)


if gdf.crs is None:

    raise ValueError(
        "Study boundary has no CRS."
    )


# ============================================================
# 13. CONVERT BOUNDARY TO WGS84
# ============================================================

try:

    gdf = gdf.to_crs(
        WGS84_PROJ
    )

except Exception as e:

    raise RuntimeError(

        "Boundary reprojection failed.\n"
        f"Original error:\n{e}"
    )


gdf = gdf[

    gdf.geometry.notna()

    &

    (~gdf.geometry.is_empty)

].copy()


if gdf.empty:

    raise ValueError(
        "Boundary contains no valid geometry."
    )


# ============================================================
# 14. TARGET GRID
# ============================================================

minx, miny, maxx, maxy = (
    gdf.total_bounds
)


left = (

    math.floor(

        minx
        /
        TARGET_RES_DEG

    )

    * TARGET_RES_DEG
)


right = (

    math.ceil(

        maxx
        /
        TARGET_RES_DEG

    )

    * TARGET_RES_DEG
)


bottom = (

    math.floor(

        miny
        /
        TARGET_RES_DEG

    )

    * TARGET_RES_DEG
)


top = (

    math.ceil(

        maxy
        /
        TARGET_RES_DEG

    )

    * TARGET_RES_DEG
)


width = int(

    round(

        (right - left)

        /

        TARGET_RES_DEG

    )
)


height = int(

    round(

        (top - bottom)

        /

        TARGET_RES_DEG

    )
)


transform = from_origin(

    left,
    top,

    TARGET_RES_DEG,
    TARGET_RES_DEG

)


print(
    "\nTarget grid:"
)

print(
    "Width :",
    width
)

print(
    "Height:",
    height
)

print(
    "Bounds:",
    (
        left,
        bottom,
        right,
        top
    )
)

print(
    "Grid spacing:",
    TARGET_RES_DEG,
    "degree"
)


# ============================================================
# 15. KHULNA INSIDE MASK
# ============================================================

inside_mask = geometry_mask(

    gdf.geometry,

    transform=transform,

    invert=True,

    out_shape=(
        height,
        width
    ),

    all_touched=False

)


print(
    "Pixels inside Khulna:",
    int(
        inside_mask.sum()
    )
)


# ============================================================
# 16. PREPARE SOURCES
# ============================================================

sources = {}


# ============================================================
# 16A. PRECIPITATION — 2022
# ============================================================

for (
    product,
    folder_name
) in PRECIP_FOLDERS.items():


    folder = (

        RAW_DIR
        /
        "precipitation"
        /
        folder_name

    )


    if not folder.exists():

        raise FileNotFoundError(

            f"Missing precipitation folder:\n"
            f"{folder}"
        )


    mm = monthly_map(
        folder
    )


    for month in range(
        1,
        13
    ):


        ym = (
            TEST_YEAR,
            month
        )


        if ym not in mm:

            raise FileNotFoundError(

                f"Missing {product} raster for "
                f"{TEST_YEAR}-{month:02d}"
            )


        sources[
            (
                product,
                month
            )
        ] = mm[
            ym
        ]


# ============================================================
# 16B. DYNAMIC LAND VARIABLES — 2022
# ============================================================

for pred in LAND_DYNAMIC:


    folder = (

        RAW_DIR
        /
        "predictors"
        /
        pred

    )


    if not folder.exists():

        raise FileNotFoundError(

            f"Missing predictor folder:\n"
            f"{folder}"
        )


    mm = monthly_map(
        folder
    )


    for month in range(
        1,
        13
    ):


        ym = (
            TEST_YEAR,
            month
        )


        if ym not in mm:

            raise FileNotFoundError(

                f"Missing {pred} raster for "
                f"{TEST_YEAR}-{month:02d}"
            )


        sources[
            (
                pred,
                month
            )
        ] = mm[
            ym
        ]


# ============================================================
# 16C. STATIC VARIABLES
# ============================================================

sources[
    (
        "DEM",
        0
    )
] = DEM_PATH


sources[
    (
        "Distance_Sea",
        0
    )
] = DFS_PATH


# ============================================================
# 17. SOURCE COUNT
# ============================================================

EXPECTED_SOURCES = 122


print(
    "\n========================================"
)

print(
    "SOURCE PREPARATION COMPLETE"
)

print(
    "========================================"
)

print(
    "Total sources prepared:",
    len(
        sources
    )
)

print(
    "Expected sources:",
    EXPECTED_SOURCES
)


if len(
    sources
) == EXPECTED_SOURCES:

    print(
        "All 122 required sources found."
    )

else:

    raise ValueError(

        f"Expected {EXPECTED_SOURCES} sources, "
        f"but found {len(sources)}."
    )


# ============================================================
# 18. OUTPUT DIRECTORY
# ============================================================

aligned_root = (

    PROCESSED_DIR
    /
    "test2022_target_grid"

)


aligned_root.mkdir(

    parents=True,
    exist_ok=True

)


# ============================================================
# 19. OUTPUT PROFILE
# ============================================================

profile = {

    "driver":
        "GTiff",

    "height":
        height,

    "width":
        width,

    "count":
        1,

    "dtype":
        "float32",

    "crs":
        WGS84_PROJ,

    "transform":
        transform,

    "nodata":
        DST_NODATA,

    "compress":
        "deflate",

    "predictor":
        3

}


# ============================================================
# 20. ALIGN ONE RASTER
# ============================================================

def align_one(
    src_path,
    out_path
):


    out_path.parent.mkdir(

        parents=True,
        exist_ok=True

    )


    dst = np.full(

        (
            height,
            width
        ),

        DST_NODATA,

        dtype=np.float32

    )


    with rasterio.open(
        src_path
    ) as src:


        # ----------------------------------------------------
        # Fix / resolve CRS
        # ----------------------------------------------------

        src_crs = source_crs_or_verified_wgs84(

            src,
            src_path

        )


        # ----------------------------------------------------
        # Read raster safely
        # ----------------------------------------------------

        src_arr = src.read(

            1,

            masked=True

        ).astype(
            np.float32
        )


        TEMP_SRC_NODATA = -9999.0


        src_data = src_arr.filled(

            TEMP_SRC_NODATA

        ).astype(
            np.float32
        )


        src_data[

            ~np.isfinite(
                src_data
            )

        ] = TEMP_SRC_NODATA


        # ----------------------------------------------------
        # Reproject / resample
        # ----------------------------------------------------

        reproject(

            source=
                src_data,

            destination=
                dst,

            src_transform=
                src.transform,

            src_crs=
                src_crs,

            src_nodata=
                TEMP_SRC_NODATA,

            dst_transform=
                transform,

            dst_crs=
                WGS84_PROJ,

            dst_nodata=
                DST_NODATA,

            resampling=
                Resampling.bilinear,

            init_dest_nodata=
                True

        )


    # ========================================================
    # MASK OUTSIDE STUDY AREA
    # ========================================================

    dst[
        ~inside_mask
    ] = DST_NODATA


    # ========================================================
    # VALID PIXELS
    # ========================================================

    valid = (

        inside_mask

        &

        np.isfinite(
            dst
        )

        &

        (
            ~np.isclose(
                dst,
                DST_NODATA
            )
        )

    )


    # ========================================================
    # WRITE RASTER
    # ========================================================

    with rasterio.open(

        out_path,
        "w",
        **profile

    ) as out:


        out.write(
            dst,
            1
        )


    # ========================================================
    # QC
    # ========================================================

    total_inside = int(

        inside_mask.sum()

    )


    valid_inside = int(

        valid.sum()

    )


    if total_inside > 0:

        valid_inside_pct = (

            valid_inside
            /
            total_inside

        ) * 100.0

    else:

        valid_inside_pct = np.nan


    if valid_inside > 0:

        values = dst[
            valid
        ]


        min_value = float(

            np.nanmin(
                values
            )
        )


        max_value = float(

            np.nanmax(
                values
            )
        )


        mean_value = float(

            np.nanmean(
                values
            )
        )


    else:

        min_value = np.nan

        max_value = np.nan

        mean_value = np.nan


    return {

        "source":
            str(
                src_path
            ),

        "output":
            str(
                out_path
            ),

        "valid_inside_pixels":
            valid_inside,

        "total_inside_pixels":
            total_inside,

        "valid_inside_pct":
            valid_inside_pct,

        "min":
            min_value,

        "max":
            max_value,

        "mean":
            mean_value

    }


# ============================================================
# 21. PROCESS ALL SOURCES
# ============================================================

qc_rows = []


for (
    name,
    month
), src_path in sources.items():


    # --------------------------------------------------------
    # Static
    # --------------------------------------------------------

    if month == 0:


        out_path = (

            aligned_root

            /
            "static"

            /
            f"{name}.tif"

        )


        label = (
            f"{name} STATIC"
        )


    # --------------------------------------------------------
    # Monthly
    # --------------------------------------------------------

    else:


        out_path = (

            aligned_root

            /
            name

            /
            f"{name}_{TEST_YEAR}_{month:02d}.tif"

        )


        label = (

            f"{name} "
            f"{TEST_YEAR}-{month:02d}"

        )


    print(
        "Processing:",
        label
    )


    result = align_one(

        src_path,
        out_path

    )


    qc_rows.append(
        result
    )


# ============================================================
# 22. QC TABLE
# ============================================================

qc = pd.DataFrame(
    qc_rows
)


display(
    qc
)


# ============================================================
# 23. SAVE QC
# ============================================================

qc_path = (

    aligned_root

    /
    "alignment_qc.csv"

)


qc.to_csv(

    qc_path,

    index=False

)


# ============================================================
# 24. COVERAGE CHECK
# ============================================================

low = qc[

    qc[
        "valid_inside_pct"
    ]

    < 95

]


if len(
    low
) > 0:


    print(

        "\nWARNING:"
        " Some rasters have less than "
        "95% valid coverage inside Khulna."

    )


    print(

        "DO NOT fill these gaps using "
        "nearest-neighbour values."

    )


    display(

        low[

            [
                "source",
                "valid_inside_pct",
                "min",
                "max",
                "mean"
            ]

        ]

    )


else:


    print(

        "\nAll target-grid inputs "
        "have >=95% valid coverage."

    )


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print(
    "\n=========================================="
)

print(
    "2022 TARGET GRID ALIGNMENT COMPLETE"
)

print(
    "=========================================="
)

print(
    "Total processed rasters:",
    len(
        qc
    )
)

print(
    "Expected rasters:",
    EXPECTED_SOURCES
)

print(
    "QC file:",
    qc_path
)

print(
    "Aligned rasters saved to:",
    aligned_root
)


if len(
    qc
) == EXPECTED_SOURCES:


    print(

        "\nSUCCESS:"
        " All 122 required 2022 input "
        "rasters were processed."

    )


else:


    print(

        "\nWARNING:"
        f" Expected {EXPECTED_SOURCES} rasters, "
        f"but processed {len(qc)}."

    )

PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh

Canonical DEM:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\DEM\Khulna_SRTM_DEM.tif

Canonical Distance to Sea:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\Distance_Sea\Distance_Sea.tif

Boundary file:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\boundary\Khulna.shp

Target grid:
Width : 105
Height: 271
Bounds: (89.235, 21.66, 89.76, 23.015)
Grid spacing: 0.005 degree
Pixels inside Khulna: 15636

SOURCE PREPARATION COMPLETE
Total sources prepared: 122
Expected sources: 122
All 122 required sources found.
Processing: CCS 2022-01
Processing: CCS 2022-02

,source,output,valid_inside_pixels,total_inside_pixels,valid_inside_pct,min,max,mean
0,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,0.000000,26.871094,4.742375
1,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,0.000000,19.937500,2.048472
2,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,0.000000,10.222656,0.965554
3,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,0.000000,12.523438,1.975626
4,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,29.000000,145.164062,79.835892
...,...,...,...,...,...,...,...,...
117,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14008,15636,89.588130,24.428797,32.294632,27.784445
118,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14008,15636,89.588130,24.507010,30.291086,25.652864
119,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14008,15636,89.588130,20.978458,25.605762,22.851723
120,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,15636,15636,100.000000,-0.000016,15.279215,4.859379



DO NOT fill these gaps using nearest-neighbour values.


,source,valid_inside_pct,min,max,mean
0,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,0.000000,26.871094,4.742375
1,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,0.000000,19.937500,2.048472
2,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,0.000000,10.222656,0.965554
3,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,0.000000,12.523438,1.975626
4,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,29.000000,145.164062,79.835892
...,...,...,...,...,...
116,E:\Geospatial\Precipitation-Downscaling-Khulna...,88.788693,21.949150,35.016968,28.167221
117,E:\Geospatial\Precipitation-Downscaling-Khulna...,89.588130,24.428797,32.294632,27.784445
118,E:\Geospatial\Precipitation-Downscaling-Khulna...,89.588130,24.507010,30.291086,25.652864
119,E:\Geospatial\Precipitation-Downscaling-Khulna...,89.588130,20.978458,25.605762,22.851723



2022 TARGET GRID ALIGNMENT COMPLETE
Total processed rasters: 122
Expected rasters: 122
QC file: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\test2022_target_grid\alignment_qc.csv
Aligned rasters saved to: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\test2022_target_grid

SUCCESS: All 122 required 2022 input rasters were processed.
